In [4]:
!pip install -q torch torch-geometric pandas scikit-learn tqdm networkx joblib

In [6]:


import os, json, math, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib
import networkx as nx

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, BatchNorm


In [13]:
# Change to your CSV path if needed:
CSV_PATH = "uk_logistics_dataset_5000_routes_20250710_124423.csv"

df = pd.read_csv(CSV_PATH)
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=['origin_name','destination_name','waypoints_json'])
print("Rows loaded:", len(df))
print("Columns:", list(df.columns)[:40])
df.head(3)


Rows loaded: 5000
Columns: ['route_id', 'origin_name', 'origin_location', 'origin_latitude', 'origin_longitude', 'destination_name', 'destination_location', 'destination_latitude', 'destination_longitude', 'waypoints_json', 'waypoint_count', 'distance_km', 'estimated_driving_time_hours', 'total_time_hours', 'breaks_required', 'avg_speed_kmh', 'vehicle_type', 'vehicle_max_weight_kg', 'vehicle_height_m', 'vehicle_width_m', 'vehicle_length_m', 'cargo_volume_m3', 'euro_standard', 'cargo_weight_kg', 'load_factor', 'cargo_type', 'fuel_consumption_liters', 'co2_emissions_kg', 'nox_emissions_g', 'pm_emissions_g', 'emissions_per_km_kg', 'fuel_cost_gbp', 'driver_cost_gbp', 'vehicle_cost_gbp', 'insurance_cost_gbp', 'maintenance_cost_gbp', 'caz_cost_gbp', 'toll_cost_gbp', 'hgv_levy_gbp', 'lez_cost_gbp']


,route_id,origin_name,origin_location,origin_latitude,origin_longitude,destination_name,destination_location,destination_latitude,destination_longitude,waypoints_json,...,month,day_of_week,is_peak_season,is_weekend,route_compliant,bridge_restrictions,weight_restrictions,data_source,generation_timestamp,dataset_version
0,RT000001,Glasgow Central,Glasgow,55.8642,-4.2518,Liverpool Port,Liverpool,53.4053,-2.9977,"[{""lat"": 54.4033, ""lon"": -1.6622, ""name"": ""A1 ...",...,6,7,No,Yes,Yes,NaN,NaN,Synthetic - UK Logistics Research,2025-07-10T12:44:22.427880,2
1,RT000002,Newcastle Hub,Newcastle,54.9783,-1.6178,Leeds Central,Leeds,53.8008,-1.5491,"[{""lat"": 53.9444, ""lon"": -1.3847, ""name"": ""Wet...",...,9,3,No,No,Yes,NaN,NaN,Synthetic - UK Logistics Research,2025-07-10T12:44:22.428054,2
2,RT000003,Portsmouth Port,Portsmouth,50.8025,-1.1088,iPort Doncaster,Doncaster,53.4939,-1.0042,"[{""lat"": 52.6167, ""lon"": -1.1667, ""name"": ""Lei...",...,3,6,No,Yes,Yes,NaN,18t limit,Synthetic - UK Logistics Research,2025-07-10T12:44:22.428211,2


In [17]:
def safe_float(x, default=None):
    try:
        if pd.isna(x): return default
        return float(x)
    except Exception:
        return default

def safe_num(x, default=0.0):
    v = safe_float(x, default)
    return default if v is None else v

def parse_waypoints(wp_json):
    if pd.isna(wp_json) or str(wp_json).strip()=="":
        return []
    if isinstance(wp_json, list):
        return wp_json
    try:
        return json.loads(wp_json)
    except Exception:
        return []

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    dlat = math.radians(lat2-lat1)
    dlon = math.radians(lon2-lon1)
    a = (math.sin(dlat/2)**2 +
         math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2)
    return 2*R*math.asin(math.sqrt(a))


In [19]:
# Collect nodes with best-known lat/lon + meta for node features
node_meta = {}  # name -> dict

for _, r in df.iterrows():
    origin = str(r['origin_name']).strip()
    dest   = str(r['destination_name']).strip()
    wps    = parse_waypoints(r.get('waypoints_json', ""))

    # origin/destination coords (support both naming variants)
    o_lat = safe_float(r.get('origin_latitude', r.get('origin_lat', None)))
    o_lon = safe_float(r.get('origin_longitude', r.get('origin_lon', None)))
    d_lat = safe_float(r.get('destination_latitude', r.get('destination_lat', None)))
    d_lon = safe_float(r.get('destination_longitude', r.get('destination_lon', None)))

    avg_speed = safe_num(r.get('avg_speed_kmh'), 60)
    weather   = str(r.get('weather_condition', 'Unknown'))
    month     = int(safe_num(r.get('month'), 0))
    dow       = int(safe_num(r.get('day_of_week'), 0))
    is_peak   = 1 if str(r.get('is_peak_season','No')).strip().lower() == 'yes' else 0
    is_wkend  = 1 if str(r.get('is_weekend','No')).strip().lower() == 'yes' else 0

    node_meta.setdefault(origin, {}).update({
        'lat': node_meta.get(origin,{}).get('lat', o_lat),
        'lon': node_meta.get(origin,{}).get('lon', o_lon),
        'avg_speed_kmh': avg_speed,
        'weather': weather,
        'month': month, 'day_of_week': dow,
        'is_peak_season': is_peak, 'is_weekend': is_wkend,
        'loc_type': str(r.get('origin_loc_type','origin'))
    })
    node_meta.setdefault(dest, {}).update({
        'lat': node_meta.get(dest,{}).get('lat', d_lat),
        'lon': node_meta.get(dest,{}).get('lon', d_lon),
        'avg_speed_kmh': avg_speed,
        'weather': weather,
        'month': month, 'day_of_week': dow,
        'is_peak_season': is_peak, 'is_weekend': is_wkend,
        'loc_type': str(r.get('destination_loc_type','destination'))
    })
    for wp in wps:
        nm = str(wp.get('name','')).strip()
        if not nm: continue
        lat = safe_float(wp.get('lat')); lon = safe_float(wp.get('lon'))
        node_meta.setdefault(nm, {}).update({
            'lat': node_meta.get(nm,{}).get('lat', lat),
            'lon': node_meta.get(nm,{}).get('lon', lon),
            'avg_speed_kmh': avg_speed,
            'weather': weather,
            'month': month, 'day_of_week': dow,
            'is_peak_season': is_peak, 'is_weekend': is_wkend,
            'loc_type': node_meta.get(nm,{}).get('loc_type','waypoint')
        })

location_list = sorted(node_meta.keys())
location2idx  = {n:i for i,n in enumerate(location_list)}

# Fit & save one-hot encoders
weather_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
loc_type_enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
weather_enc.fit(np.array([node_meta[n]['weather'] for n in location_list]).reshape(-1,1))
loc_type_enc.fit(np.array([node_meta[n]['loc_type'] for n in location_list]).reshape(-1,1))
joblib.dump(weather_enc, "weather_enc.pkl")
joblib.dump(loc_type_enc, "loc_type_enc.pkl")

# Node features: base + one-hots
node_features = []
for n in location_list:
    m = node_meta[n]
    base = [
        safe_num(m['lat'], 0.0), safe_num(m['lon'], 0.0),
        safe_num(m['avg_speed_kmh'], 60.0),
        safe_num(m['month'], 0), safe_num(m['day_of_week'], 0),
        safe_num(m['is_peak_season'], 0), safe_num(m['is_weekend'], 0)
    ]
    w = weather_enc.transform([[m['weather']]])[0].astype(float).tolist()
    lt = loc_type_enc.transform([[m['loc_type']]])[0].astype(float).tolist()
    node_features.append(base + w + lt)

node_features = np.array(node_features, dtype=np.float32)
print("node_features shape:", node_features.shape)


node_features shape: (68, 15)


In [21]:
objective_list = ['cheapest','fastest','greenest']

edge_cont_cols = [
    'distance_km',
    'estimated_driving_time_hours',
    'total_time_hours',
    'avg_speed_kmh',
    'co2_emissions_kg',
    'nox_emissions_g', 'pm_emissions_g', 'emissions_per_km_kg',
    'fuel_cost_gbp', 'total_cost_gbp', 'cost_per_km_gbp',
    'traffic_delay_minutes','weather_delay_minutes',
    'vehicle_max_weight_kg',
    'vehicle_length_m','vehicle_width_m','vehicle_height_m',
    'cargo_volume_m3','cargo_weight_kg','load_factor',
]
edge_bin_cols = ['is_peak_season','is_weekend']  # 0/1

label_map = {
    'cheapest': 'total_cost_gbp',
    'fastest':  'total_time_hours',
    'greenest': 'co2_emissions_kg',
}

edge_data_by_obj = {obj: [] for obj in objective_list}

for _, r in df.iterrows():
    origin = str(r['origin_name']).strip()
    dest   = str(r['destination_name']).strip()
    wps    = parse_waypoints(r.get('waypoints_json', ""))

    chain = [origin] + [str(w.get('name','')).strip() for w in wps if str(w.get('name','')).strip()] + [dest]
    feat_cont = [safe_num(r.get(c), 0.0) for c in edge_cont_cols]
    feat_bin  = [1 if str(r.get(c,'No')).strip().lower()=='yes' else 0 for c in edge_bin_cols]
    edge_feat = np.array(feat_cont + feat_bin, dtype=np.float32)

    for obj in objective_list:
        label = safe_num(r.get(label_map[obj]), 0.0)
        for i in range(len(chain)-1):
            src = location2idx.get(chain[i]); dst = location2idx.get(chain[i+1])
            if src is None or dst is None: continue
            edge_data_by_obj[obj].append({
                'src': src, 'dst': dst,
                'features': edge_feat.copy(),
                'label': label
            })

for obj in objective_list:
    print(f"{obj}: {len(edge_data_by_obj[obj])} edges")


cheapest: 15179 edges
fastest: 15179 edges
greenest: 15179 edges


In [23]:
num_cont_feats = len(edge_cont_cols)  # normalize only continuous part
scalers = {}
for obj in objective_list:
    E = edge_data_by_obj[obj]
    feat = np.stack([e['features'] for e in E], axis=0)
    lab  = np.array([e['label'] for e in E], dtype=np.float32)

    scaler = StandardScaler()
    feat[:, :num_cont_feats] = scaler.fit_transform(feat[:, :num_cont_feats])
    scalers[obj] = scaler
    joblib.dump(scaler, f"scaler_{obj}.pkl")

    for i, e in enumerate(E):
        e['features'] = feat[i]

    print(f"\n=== {obj} ===")
    print("Feature NaN?", np.isnan(feat).any(), "| Label NaN?", np.isnan(lab).any())
    print("Feature min/max:", np.nanmin(feat), np.nanmax(feat))
    print("Label  min/max:", np.nanmin(lab),  np.nanmax(lab))



=== cheapest ===
Feature NaN? False | Label NaN? False
Feature min/max: -2.223503 29.765833
Label  min/max: 16.78 2202.51

=== fastest ===
Feature NaN? False | Label NaN? False
Feature min/max: -2.223503 29.765833
Label  min/max: 0.05 21.36

=== greenest ===
Feature NaN? False | Label NaN? False
Feature min/max: -2.223503 29.765833
Label  min/max: 2.78 981.6


In [25]:
class DeeperEdgeGNN(nn.Module):
    def __init__(self, node_in, edge_in, hidden_dim=128, num_layers=6, p_drop=0.30):
        super().__init__()
        self.gcn_layers = nn.ModuleList()
        self.bn_layers  = nn.ModuleList()
        self.gcn_layers.append(GCNConv(node_in, hidden_dim))
        self.bn_layers.append(BatchNorm(hidden_dim))
        for _ in range(num_layers-1):
            self.gcn_layers.append(GCNConv(hidden_dim, hidden_dim))
            self.bn_layers.append(BatchNorm(hidden_dim))
        self.dropout = nn.Dropout(p_drop)
        self.fc_edge = nn.Sequential(
            nn.Linear(hidden_dim*2 + edge_in, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(p_drop),
            nn.Linear(256, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(p_drop),
            nn.Linear(64, 1)
        )
    def forward(self, data: Data):
        x = data.x
        for conv, bn in zip(self.gcn_layers, self.bn_layers):
            x = torch.relu(bn(conv(x, data.edge_index)))
            x = self.dropout(x)
        src = x[data.edge_index[0]]
        dst = x[data.edge_index[1]]
        edge_feat = torch.cat([src, dst, data.edge_attr], dim=1)
        return self.fc_edge(edge_feat).squeeze(-1)


In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
x = torch.tensor(node_features, dtype=torch.float, device=device)

num_folds     = 5
epochs        = 2000
hidden_dim    = 128
num_layers    = 6
lr            = 2e-3
weight_decay  = 2e-3
val_frac      = 0.15

model_paths = {}
fold_losses = {obj: {'train': [], 'val': [], 'test': []} for obj in objective_list}

for obj in objective_list:
    print(f"\n=== Training: {obj} ===")
    data_obj  = edge_data_by_obj[obj]
    edge_idx  = torch.tensor([[e['src'] for e in data_obj],
                              [e['dst'] for e in data_obj]], dtype=torch.long, device=device)
    edge_attr = torch.tensor(np.stack([e['features'] for e in data_obj]), dtype=torch.float, device=device)
    labels    = torch.tensor([e['label'] for e in data_obj], dtype=torch.float, device=device)
    data      = Data(x=x, edge_index=edge_idx, edge_attr=edge_attr, y=labels)

    best_model_state = None
    best_model_val = float('inf')

    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
    idx_all = np.arange(labels.shape[0])

    for fold, (trainval_idx, test_idx) in enumerate(kf.split(idx_all), 1):
        tr_idx, val_idx = train_test_split(trainval_idx, test_size=val_frac, random_state=fold)
        tr_idx  = torch.tensor(tr_idx,  dtype=torch.long, device=device)
        val_idx = torch.tensor(val_idx, dtype=torch.long, device=device)
        te_idx  = torch.tensor(test_idx, dtype=torch.long, device=device)

        model = DeeperEdgeGNN(x.shape[1], edge_attr.shape[1],
                              hidden_dim=hidden_dim, num_layers=num_layers).to(device)
        opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        crit = nn.MSELoss()
        sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=30, min_lr=1e-5)

        best_val = float('inf'); best_train = None; best_state = None

        for ep in range(epochs):
            model.train()
            opt.zero_grad()
            pred = model(data)
            loss = crit(pred[tr_idx], labels[tr_idx])
            loss.backward()
            opt.step()

            if (ep % 10 == 0) or (ep == epochs-1):
                model.eval()
                with torch.no_grad():
                    v_pred = model(data)[val_idx]
                    v_loss = crit(v_pred, labels[val_idx])
                sch.step(v_loss)
                print(f"Obj: {obj} | Fold {fold} | Epoch {ep}: Train {loss.item():.4f}  Val {v_loss.item():.4f}")
                if v_loss.item() < best_val:
                    best_val = v_loss.item()
                    best_train = loss.item()
                    best_state = model.state_dict()

        # Test with best fold state
        model.load_state_dict(best_state); model.eval()
        with torch.no_grad():
            t_pred = model(data)[te_idx]
            t_loss = crit(t_pred, labels[te_idx]).item()

        fold_losses[obj]['train'].append(best_train)
        fold_losses[obj]['val'].append(best_val)
        fold_losses[obj]['test'].append(t_loss)
        print(f"Fold {fold}: Best Train {best_train:.4f} | Best Val {best_val:.4f} | Test {t_loss:.4f}")

        # Track the overall best state across folds
        if best_val < best_model_val:
            best_model_val = best_val
            best_model_state = best_state

    # Save best across folds
    model_path = f"best_gnn_{obj}.pt"
    torch.save(best_model_state, model_path)
    model_paths[obj] = model_path
    print(f"Saved best model for {obj} → {model_path}")

# Summary
print('\n====== NESTED K-FOLD CROSS-VALIDATION SUMMARY ======')
print('{:<10} {:<25} {:<25} {:<25}'.format('Obj', 'Avg Train MSE (±std)', 'Avg Val MSE (±std)', 'Avg Test MSE (±std)'))
for obj in objective_list:
    tr = np.array(fold_losses[obj]['train']); vl = np.array(fold_losses[obj]['val']); te = np.array(fold_losses[obj]['test'])
    print('{:<10} {:<25} {:<25} {:<25}'.format(
        obj,
        f"{tr.mean():.2f} ± {tr.std():.2f}",
        f"{vl.mean():.2f} ± {vl.std():.2f}",
        f"{te.mean():.2f} ± {te.std():.2f}",
    ))


Using device: cpu

=== Training: cheapest ===
Obj: cheapest | Fold 1 | Epoch 0: Train 779561.3750  Val 775756.1250
Obj: cheapest | Fold 1 | Epoch 10: Train 776494.6875  Val 778656.8750
Obj: cheapest | Fold 1 | Epoch 20: Train 775057.4375  Val 771500.8125
Obj: cheapest | Fold 1 | Epoch 30: Train 773515.2500  Val 768529.5000
Obj: cheapest | Fold 1 | Epoch 40: Train 771749.2500  Val 768366.5000
Obj: cheapest | Fold 1 | Epoch 50: Train 769684.2500  Val 767969.5625
Obj: cheapest | Fold 1 | Epoch 60: Train 767278.1875  Val 764776.1875
Obj: cheapest | Fold 1 | Epoch 70: Train 764616.8750  Val 766306.6250
Obj: cheapest | Fold 1 | Epoch 80: Train 761770.2500  Val 760703.9375
Obj: cheapest | Fold 1 | Epoch 90: Train 758673.3125  Val 764236.1250
Obj: cheapest | Fold 1 | Epoch 100: Train 755187.6250  Val 758359.6250
Obj: cheapest | Fold 1 | Epoch 110: Train 751545.1875  Val 695263.8125
Obj: cheapest | Fold 1 | Epoch 120: Train 747357.7500  Val 672328.0000
Obj: cheapest | Fold 1 | Epoch 130: Train 

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x_t = torch.tensor(node_features, dtype=torch.float, device=device)

def precompute_edge_predictions(obj, data_obj):
    edge_idx  = torch.tensor([[e['src'] for e in data_obj],
                              [e['dst'] for e in data_obj]], dtype=torch.long, device=device)
    edge_attr = torch.tensor(np.stack([e['features'] for e in data_obj]), dtype=torch.float, device=device)
    labels    = torch.tensor([e['label'] for e in data_obj], dtype=torch.float, device=device)
    data      = Data(x=x_t, edge_index=edge_idx, edge_attr=edge_attr, y=labels)

    model = DeeperEdgeGNN(x_t.shape[1], edge_attr.shape[1]).to(device)
    model.load_state_dict(torch.load(f"best_gnn_{obj}.pt", map_location=device))
    model.eval()

    with torch.no_grad():
        preds = model(data).detach().cpu().numpy()

    # Non-negative, tiny epsilon for stability
    preds = np.maximum(preds, 0.0) + 1e-9

    # Mean per (src,dst)
    from collections import defaultdict
    agg = defaultdict(list)
    srcs = edge_idx[0].detach().cpu().numpy()
    dsts = edge_idx[1].detach().cpu().numpy()
    for s, d, p in zip(srcs, dsts, preds):
        agg[(int(s), int(d))].append(float(p))

    uv_pred = {k: float(np.mean(v)) for k, v in agg.items()}
    return uv_pred

pred_maps = {}
for obj in objective_list:
    print(f"Precomputing predictions: {obj}")
    pred_maps[obj] = precompute_edge_predictions(obj, edge_data_by_obj[obj])
    print(f"Unique predicted edges ({obj}):", len(pred_maps[obj]))


Precomputing predictions: cheapest
Unique predicted edges (cheapest): 1706
Precomputing predictions: fastest
Unique predicted edges (fastest): 1706
Precomputing predictions: greenest
Unique predicted edges (greenest): 1706


In [31]:
def build_pred_graph(obj, pred_map):
    G = nx.DiGraph()
    # Add nodes with coords
    for n in location_list:
        info = node_meta[n]
        G.add_node(n, lat=info.get('lat'), lon=info.get('lon'))
    # Add predicted edges
    for (s, d), w in pred_map.items():
        u = location_list[s]; v = location_list[d]
        G.add_edge(u, v, weight=float(w))
    return G

G_pred = {obj: build_pred_graph(obj, pred_maps[obj]) for obj in objective_list}
for obj in objective_list:
    print(obj, "→ nodes:", len(G_pred[obj].nodes), "edges:", len(G_pred[obj].edges))


cheapest → nodes: 68 edges: 1706
fastest → nodes: 68 edges: 1706
greenest → nodes: 68 edges: 1706


In [33]:
def haversine_for_graph(u, v, G):
    lat1, lon1 = G.nodes[u].get('lat'), G.nodes[u].get('lon')
    lat2, lon2 = G.nodes[v].get('lat'), G.nodes[v].get('lon')
    if None in (lat1, lon1, lat2, lon2): return 0.0
    return haversine_km(lat1, lon1, lat2, lon2)

def astar_heuristic_pred(u, v, G):
    # tiny admissible heuristic in predicted-weight space
    return 0.01 * haversine_for_graph(u, v, G)

def find_best_route_pred(origin, destination, objective='cheapest', method='astar', k_paths=3, verbose=True):
    assert objective in G_pred, "Invalid objective"
    G = G_pred[objective]
    if origin not in G or destination not in G:
        raise ValueError("Origin/Destination not present in graph nodes.")

    if method == 'dijkstra':
        path = nx.dijkstra_path(G, origin, destination, weight='weight')
        paths = [path]
    elif method == 'astar':
        h = lambda u,v: astar_heuristic_pred(u, v, G)
        path = nx.astar_path(G, origin, destination, heuristic=h, weight='weight')
        paths = [path]
    elif method == 'kshortest':
        gen = nx.shortest_simple_paths(G, origin, destination, weight='weight')
        paths = []
        try:
            for _ in range(k_paths):
                paths.append(next(gen))
        except StopIteration:
            pass
        if not paths:
            raise nx.NetworkXNoPath("No path(s) found")
    else:
        raise ValueError("method must be 'dijkstra' | 'astar' | 'kshortest'")

    reports = []
    for path in paths:
        edges = list(zip(path[:-1], path[1:]))
        total = sum(G[u][v]['weight'] for u,v in edges)
        rep_edges = [{'u':u,'v':v,'weight':G[u][v]['weight']} for u,v in edges]
        reports.append({'path': path, 'total_weight': total, 'edges': rep_edges})

    if verbose:
        print(f"\nBest {objective} ({method}) path:", reports[0]['path'])
        print(f"Predicted total weight: {reports[0]['total_weight']:.3f}")
        print("Edgewise predictions:")
        for e in reports[0]['edges']:
            print(f"  {e['u']} → {e['v']}: {e['weight']:.3f}")
    return reports


In [35]:
import math, json
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

def _safe_float(x, default=None):
    try:
        if pd.isna(x): return default
        return float(x)
    except Exception:
        return default

def _safe_num(x, default=0.0):
    v = _safe_float(x, default)
    return default if v is None else v

def _parse_waypoints(wp_json):
    if pd.isna(wp_json) or str(wp_json).strip()=="":
        return []
    if isinstance(wp_json, list):
        return wp_json
    try:
        return json.loads(wp_json)
    except Exception:
        return []

def _segment_nodes(row):
    """Return [(name, lat, lon), ...] origin→waypoints→destination"""
    def pick_latlon(prefix_lat, prefix_lon):
        lat = row.get(prefix_lat, row.get(prefix_lat.replace('latitude','lat')))
        lon = row.get(prefix_lon, row.get(prefix_lon.replace('longitude','lon')))
        lat = _safe_float(lat, None) if lat is not None else None
        lon = _safe_float(lon, None) if lon is not None else None
        return lat, lon

    origin = str(row['origin_name']).strip()
    dest   = str(row['destination_name']).strip()
    o_lat, o_lon = pick_latlon('origin_latitude','origin_longitude')
    d_lat, d_lon = pick_latlon('destination_latitude','destination_longitude')

    chain = [(origin, o_lat, o_lon)]
    for wp in _parse_waypoints(row.get('waypoints_json', "")):
        nm = str(wp.get('name','')).strip()
        if nm:
            chain.append((nm, _safe_float(wp.get('lat'), None), _safe_float(wp.get('lon'), None)))
    chain.append((dest, d_lat, d_lon))
    return chain

def _haversine_km(lat1, lon1, lat2, lon2):
    if None in (lat1,lon1,lat2,lon2): return 1.0  # minimal fallback to allow splitting
    R = 6371.0088
    dlat = math.radians(lat2-lat1); dlon = math.radians(lon2-lon1)
    a = (math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2)
    return max(1e-6, 2*R*math.asin(math.sqrt(a)))

def build_physical_edges(df):
    """
    Aggregate per-edge attributes from your CSV to support constraints/penalties.
    We split route totals across its segments by distance share, then average duplicates.
    Returns dict keyed by (u_name, v_name).
    """
    acc = defaultdict(lambda: Counter())

    # names we might have (all optional, handled robustly)
    has = set(df.columns)
    def getcol(row, name, default=0.0):
        return _safe_num(row[name], default) if name in has else float(default)

    for _, row in df.iterrows():
        chain = _segment_nodes(row)
        if len(chain) < 2: continue

        # segment distances (geometric)
        segs = []
        for (n1, lat1, lon1), (n2, lat2, lon2) in zip(chain[:-1], chain[1:]):
            segs.append((n1, n2, _haversine_km(lat1, lon1, lat2, lon2)))
        total_geom = sum(s[2] for s in segs)
        if total_geom <= 0: continue

        # route-level metrics
        route_dist   = getcol(row, 'distance_km', total_geom)
        route_time   = getcol(row, 'total_time_hours', getcol(row, 'estimated_driving_time_hours', 0))
        route_cost   = getcol(row, 'total_cost_gbp', getcol(row, 'fuel_cost_gbp', 0))
        route_emis   = getcol(row, 'co2_emissions_kg', 0)
        toll_total   = getcol(row, 'toll_gbp', 0)  # not all datasets have this, OK
        tr_delay_min = getcol(row, 'traffic_delay_minutes', 0)
        wx_delay_min = getcol(row, 'weather_delay_minutes', 0)

        # "route_compliant" & vehicle constraints (route-level; we take conservative/min later)
        route_compliant = 1 if str(row.get('route_compliant','Yes')).strip().lower() == 'yes' else 0
        veh_max_w_kg = getcol(row, 'vehicle_max_weight_kg', 1e9)  # large default -> unconstrained
        veh_len_m    = getcol(row, 'vehicle_length_m', 0)
        veh_wid_m    = getcol(row, 'vehicle_width_m',  0)
        veh_hgt_m    = getcol(row, 'vehicle_height_m', 0)

        # distribute by distance share
        for u, v, seg_km in segs:
            share = seg_km / total_geom
            key = (u, v)
            a = acc[key]
            a['count']          += 1
            a['distance_km']    += route_dist   * share
            a['travel_time_hr'] += route_time   * share
            a['fuel_cost_gbp']  += route_cost   * share
            a['co2_kg']         += route_emis   * share
            a['toll_gbp']       += toll_total   * share
            a['traffic_hr']     += (tr_delay_min * share) / 60.0
            a['weather_hr']     += (wx_delay_min * share) / 60.0

            # for constraints: keep conservative limits
            a['compliant_sum']  += route_compliant
            # we want the *minimum* allowed weight along this edge across all occurrences
            a['veh_max_w_kg_min'] = min(a.get('veh_max_w_kg_min', 1e9), veh_max_w_kg)
            # we keep *max* required dims seen (to be safe)
            a['veh_len_m_max']  = max(a.get('veh_len_m_max', 0), veh_len_m)
            a['veh_wid_m_max']  = max(a.get('veh_wid_m_max', 0), veh_wid_m)
            a['veh_hgt_m_max']  = max(a.get('veh_hgt_m_max', 0), veh_hgt_m)

    # average/derive edge attributes
    phys_edges = {}
    for key, a in acc.items():
        c = max(1, int(a['count']))
        dist_km   = a['distance_km']    / c
        time_hr   = a['travel_time_hr'] / c
        cost_gbp  = a['fuel_cost_gbp']  / c
        co2_kg    = a['co2_kg']         / c
        toll_gbp  = a['toll_gbp']       / c
        t_hr      = a['traffic_hr']     / c
        w_hr      = a['weather_hr']     / c
        comp_p    = a['compliant_sum']  / c

        phys_edges[key] = {
            'distance_km':       float(max(dist_km, 1e-9)),
            'travel_time_hr':    float(max(time_hr, 0.0)),
            'fuel_cost_gbp':     float(max(cost_gbp, 0.0)),
            'co2_kg':            float(max(co2_kg, 0.0)),
            'toll_gbp':          float(max(toll_gbp, 0.0)),
            'traffic_delay_hr':  float(max(t_hr, 0.0)),
            'weather_delay_hr':  float(max(w_hr, 0.0)),
            'compliance_prob':   float(np.clip(comp_p, 0.0, 1.0)),
            'vehicle_max_weight_kg_min': float(a.get('veh_max_w_kg_min', 1e9)),
            'vehicle_length_m_max':      float(a.get('veh_len_m_max', 0.0)),
            'vehicle_width_m_max':       float(a.get('veh_wid_m_max', 0.0)),
            'vehicle_height_m_max':      float(a.get('veh_hgt_m_max', 0.0)),
        }
    return phys_edges

phys_edges = build_physical_edges(df)
len(phys_edges)


1706

In [37]:
import networkx as nx

def build_constrained_graph(objective, pred_map, location_list, node_meta, phys_edges, constraints):
    """
    objective: 'cheapest' | 'fastest' | 'greenest'
    pred_map: {(src_idx, dst_idx): predicted_weight}
    phys_edges: {(u_name, v_name): attributes from build_physical_edges}
    constraints: dict with keys (all optional):
        - require_compliant: bool
        - compliance_threshold: float in [0,1]
        - max_vehicle_weight_kg: float or None
        - max_vehicle_height_m / width_m / length_m: float or None
        - avoid_tolls: bool
        - toll_penalty: float (large), applied if avoid_tolls=True and toll>0
        - toll_penalty_per_gbp: float, soft penalty proportional to toll_gbp
        - traffic_weight: float, add traffic_weight * traffic_delay_hr
        - co2_per_km_cap: float or None, if exceeded add co2_cap_penalty*(excess)
        - co2_cap_penalty: float
        - forbid_nodes: set/list of node names to avoid (hard)
        - forbid_edges: set/list of (u,v) names to avoid (hard)
    """
    forbid_nodes = set(constraints.get('forbid_nodes', []))
    forbid_edges = set(tuple(e) for e in constraints.get('forbid_edges', []))

    G = nx.DiGraph()

    # add nodes
    for n in location_list:
        if n in forbid_nodes:
            continue
        meta = node_meta[n]
        G.add_node(n, lat=meta.get('lat'), lon=meta.get('lon'))

    # add edges with constraints
    for (s, d), w_pred in pred_map.items():
        u = location_list[s]; v = location_list[d]
        if u not in G or v not in G:
            continue
        if (u, v) in forbid_edges:
            continue

        attrs = phys_edges.get((u, v), {})
        # ----- hard filters -----
        if constraints.get('require_compliant', False):
            if attrs.get('compliance_prob', 1.0) < constraints.get('compliance_threshold', 0.7):
                continue

        mw = constraints.get('max_vehicle_weight_kg', None)
        if mw is not None:
            # forbid edge if its *min allowed* weight < required weight
            if attrs.get('vehicle_max_weight_kg_min', 1e9) < mw:
                continue

        # (Optional) simple dimension gates; skip edge if seen max "required" dims exceed your vehicle
        mh = constraints.get('max_vehicle_height_m', None)
        if mh is not None and attrs.get('vehicle_height_m_max', 0.0) > mh:
            continue
        mwid = constraints.get('max_vehicle_width_m', None)
        if mwid is not None and attrs.get('vehicle_width_m_max', 0.0) > mwid:
            continue
        mlen = constraints.get('max_vehicle_length_m', None)
        if mlen is not None and attrs.get('vehicle_length_m_max', 0.0) > mlen:
            continue

        # ----- soft penalties on top of predicted weight -----
        w = float(w_pred)  # base = GNN prediction (already clipped >= 0)

        # Tolls
        toll_gbp = float(attrs.get('toll_gbp', 0.0))
        if constraints.get('avoid_tolls', False) and toll_gbp > 0:
            w += float(constraints.get('toll_penalty', 1e6))  # large penalty to strongly avoid
        w += float(constraints.get('toll_penalty_per_gbp', 0.0)) * toll_gbp

        # Traffic sensitivity
        w += float(constraints.get('traffic_weight', 0.0)) * float(attrs.get('traffic_delay_hr', 0.0))

        # Emissions cap (per km)
        co2_kg = float(attrs.get('co2_kg', 0.0))
        dist_km = float(attrs.get('distance_km', 1e-9))
        co2_per_km = co2_kg / max(dist_km, 1e-9)
        cap = constraints.get('co2_per_km_cap', None)
        if cap is not None and co2_per_km > cap:
            w += float(constraints.get('co2_cap_penalty', 1e3)) * (co2_per_km - cap)

        # add edge
        G.add_edge(u, v, weight=w, pred_weight=float(w_pred), **attrs)

    return G


In [39]:
import networkx as nx

def best_route_constrained_concise(
    origin: str,
    destination: str,
    objective: str,
    constraints: dict,
    method: str = "astar",
    k_paths: int = 3,
    title_prefix: str = "Best"
):
    """
    Uses your constrained graph builder, finds the best path,
    and prints output in the compact format you requested.
    Shows only: path, total weight, and one line per edge with effective weight.
    """
    assert objective in pred_maps, f"Unknown objective: {objective}"
    pred_map = pred_maps[objective]

    # build constrained graph from predicted weights + physical attributes + constraints
    Gc = build_constrained_graph(
        objective=objective,
        pred_map=pred_map,
        location_list=location_list,
        node_meta=node_meta,
        phys_edges=phys_edges,
        constraints=constraints,
    )

    if origin not in Gc or destination not in Gc:
        raise ValueError("Origin/Destination not present after constraints; relax constraints or check node names.")

    # choose search method
    if method == "dijkstra":
        path = nx.dijkstra_path(Gc, origin, destination, weight="weight")
        paths = [path]
    elif method == "astar":
        # tiny admissible heuristic (re-using the same one from earlier)
        def _hav_km_graph(u, v):
            lat1, lon1 = Gc.nodes[u].get('lat'), Gc.nodes[u].get('lon')
            lat2, lon2 = Gc.nodes[v].get('lat'), Gc.nodes[v].get('lon')
            if None in (lat1, lon1, lat2, lon2):
                return 0.0
            R = 6371.0088
            from math import radians, sin, cos, asin, sqrt
            dlat = radians(lat2-lat1); dlon = radians(lon2-lon1)
            a = (sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2)
            return 2*R*asin(sqrt(a))
        h = lambda u,v: 0.01 * _hav_km_graph(u, v)
        path = nx.astar_path(Gc, origin, destination, heuristic=h, weight="weight")
        paths = [path]
    elif method == "kshortest":
        gen = nx.shortest_simple_paths(Gc, origin, destination, weight="weight")
        paths = []
        try:
            for _ in range(k_paths):
                paths.append(next(gen))
        except StopIteration:
            pass
        if not paths:
            raise nx.NetworkXNoPath("No path(s) found under constraints.")
    else:
        raise ValueError("method must be 'dijkstra' | 'astar' | 'kshortest'.")

    # we print only the first/best
    path = paths[0]
    edges = list(zip(path[:-1], path[1:]))
    total = sum(Gc[u][v]["weight"] for u, v in edges)

    # ---- compact output (same style as your screenshot) ----
    print(f"{title_prefix} {objective} ({method}) path: {path}")
    print(f"Predicted total weight: {total:.3f}")
    print("Edgewise predictions:")
    for u, v in edges:
        print(f"  {u} → {v}: {Gc[u][v]['weight']:.3f}")

    # also return data if you want to use it programmatically
    return {
        "path": path,
        "total_weight": float(total),
        "edges": [{"u": u, "v": v, "weight": float(Gc[u][v]["weight"])} for u, v in edges],
    }


In [41]:
def _hav_km_graph(u, v, G):
    lat1, lon1 = G.nodes[u].get('lat'), G.nodes[u].get('lon')
    lat2, lon2 = G.nodes[v].get('lat'), G.nodes[v].get('lon')
    if None in (lat1, lon1, lat2, lon2): return 0.0
    R = 6371.0088
    dlat = math.radians(lat2-lat1); dlon = math.radians(lon2-lon1)
    a = (math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2)
    return 2*R*math.asin(math.sqrt(a))

def _astar_h(u, v, G):
    # tiny admissible heuristic
    return 0.01 * _hav_km_graph(u, v, G)

def find_route_with_constraints(origin, destination, objective, pred_map, location_list, node_meta, phys_edges, constraints, method='astar', k_paths=3, verbose=True):
    G = build_constrained_graph(objective, pred_map, location_list, node_meta, phys_edges, constraints)
    if origin not in G or destination not in G:
        raise ValueError("Origin/Destination not present after constraints; try relaxing them.")

    if method == 'dijkstra':
        path = nx.dijkstra_path(G, origin, destination, weight='weight')
        paths = [path]
    elif method == 'astar':
        path = nx.astar_path(G, origin, destination, heuristic=lambda u,v: _astar_h(u,v,G), weight='weight')
        paths = [path]
    elif method == 'kshortest':
        gen = nx.shortest_simple_paths(G, origin, destination, weight='weight')
        paths = []
        try:
            for _ in range(k_paths):
                paths.append(next(gen))
        except StopIteration:
            pass
        if not paths:
            raise nx.NetworkXNoPath("No path(s) found under constraints.")
    else:
        raise ValueError("method must be 'dijkstra'|'astar'|'kshortest'.")

    # Report best
    reports = []
    for path in paths:
        edges = list(zip(path[:-1], path[1:]))
        total = sum(G[u][v]['weight'] for u,v in edges)
        detail = []
        for u,v in edges:
            e = G[u][v]
            detail.append({
                'u':u,'v':v,
                'final_weight':e['weight'],
                'pred_weight':e.get('pred_weight',np.nan),
                'toll_gbp':e.get('toll_gbp',0.0),
                'co2_kg':e.get('co2_kg',0.0),
                'distance_km':e.get('distance_km',0.0),
                'traffic_delay_hr':e.get('traffic_delay_hr',0.0),
                'compliance_prob':e.get('compliance_prob',1.0),
                'vehicle_max_weight_kg_min':e.get('vehicle_max_weight_kg_min',1e9),
            })
        reports.append({'path': path, 'total_weight': total, 'edges': detail})

    if verbose:
        print(f"\nBest {objective} path with constraints: {reports[0]['path']}")
        print(f"Total constrained objective: {reports[0]['total_weight']:.3f}")
        print("Edgewise (final_weight | pred_weight | toll | CO₂ | dist | traffic | compliance | max_w_kg_min):")
        for e in reports[0]['edges']:
            print(f"  {e['u']} → {e['v']}: {e['final_weight']:.3f} | {e['pred_weight']:.3f} | £{e['toll_gbp']:.2f} | {e['co2_kg']:.2f}kg | {e['distance_km']:.1f}km | {e['traffic_delay_hr']:.2f}h | {e['compliance_prob']:.2f} | {e['vehicle_max_weight_kg_min']:.0f}")
    return reports


In [43]:
!pip -q install "gymnasium>=0.29.1" "stable-baselines3[extra]>=2.3.0" "sb3-contrib>=2.3.0" "shimmy>=2.0.0"


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
catboost 1.2.7 requires numpy<2.0,>=1.16.0, but you have numpy 2.2.6 which is incompatible.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.1 which is incompatible.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
tensorflow-intel 2.18.0 requires tensorboard<2.19,>=2.18, but you have tensorboard 2.19.0 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.15.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=

In [83]:
# ============================================================
# RL ROUTING — Destination-aware masks + GNN-blended reward + Beam finisher
# Paste this AFTER Obj-2 (after G_pred & phys_edges are ready).
# If needed once:
# !pip -q install "gymnasium>=0.29.1" "stable-baselines3[extra]>=2.3.0" "sb3-contrib>=2.3.0" "shimmy>=2.0.0"
# ============================================================

import math
from typing import Dict, Tuple, Optional, List
from collections import defaultdict, deque

import numpy as np
import networkx as nx
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.env_util import make_vec_env
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers.action_masker import ActionMasker

# ----- knobs you can tweak (sensible defaults) -------------------------------
OBJECTIVES   = ["cheapest", "fastest", "greenest"]
BLEND_ALPHA  = 0.20     # 0 = KPI-only; 0.2 = small GNN influence; 1 = GNN-only
SHAPING_ETA  = 0.35     # tiny potential shaping toward destination
STEP_PENALTY = 0.04
BACKTRACK_P  = 0.22
REVISIT_P    = 0.45
GOAL_BONUS   = 2.0
REVISIT_LIMIT = 1       # end episode if a node is visited > this
BEAM_K       = 5        # finisher beam width
BEAM_STEPS   = 200      # max extra steps finisher may add

# ----------------------------- cost helpers ----------------------------------
def edge_cost_components(phys_edges, u, v):
    pe = phys_edges.get((u, v), {})
    # time (hr) includes delays
    time_hr = float(pe.get("travel_time_hr", 0.0)) \
            + float(pe.get("traffic_delay_hr", 0.0)) \
            + float(pe.get("weather_delay_hr", 0.0))
    # money (£): prefer total_cost_gbp; otherwise sum components
    total_cost = float(pe.get("total_cost_gbp", 0.0))
    if total_cost <= 0.0:
        total_cost = (
            float(pe.get("fuel_cost_gbp", 0.0)) + float(pe.get("toll_gbp", 0.0)) +
            float(pe.get("caz_charge_gbp", 0.0)) + float(pe.get("hgv_levy_gbp", 0.0)) +
            float(pe.get("lez_cost_gbp", 0.0)) + float(pe.get("driver_cost_gbp", 0.0)) +
            float(pe.get("vehicle_cost_gbp", 0.0)) + float(pe.get("insurance_cost_gbp", 0.0)) +
            float(pe.get("maintenance_cost_gbp", 0.0))
        )
    co2 = float(pe.get("co2_kg", 0.0))
    return time_hr, total_cost, co2

def normalize_scale(G: nx.DiGraph, phys_edges, objective: str) -> float:
    vals = []
    for u, v in G.edges():
        t, m, c = edge_cost_components(phys_edges, u, v)
        vals.append( (t if objective=="fastest" else m if objective=="cheapest" else c) or 1e-6 )
    med = float(np.median(vals)) if vals else 1.0
    return max(med, 1e-6)

def precompute_phi(G: nx.DiGraph, dest: str):
    try:
        Grev = G.reverse(copy=False)
        d = nx.single_source_dijkstra_path_length(Grev, dest, weight="weight")
    except Exception:
        Grev = G.reverse(copy=False)
        d = nx.single_source_shortest_path_length(Grev, dest)
    vals = [float(x) for x in d.values() if np.isfinite(x)]
    mx = max(1.0, max(vals) if vals else 1.0)
    return {n: (float(d.get(n, np.inf))/mx if np.isfinite(d.get(n, np.inf)) else 1.0) for n in G.nodes()}

def nodes_that_can_reach_dest(G: nx.DiGraph, dest: str) -> set:
    # reverse-graph BFS from dest: all nodes that can reach dest in original graph
    Grev = G.reverse(copy=False)
    seen = {dest}
    q = deque([dest])
    while q:
        x = q.popleft()
        for nb in Grev.neighbors(x):
            if nb not in seen:
                seen.add(nb); q.append(nb)
    return seen

# ----------------------------- Environment -----------------------------------
class RoutingEnv(gym.Env):
    """
    Obs: one-hot(current) + one-hot(dest) + phi(current)  → (2N+1,)
    Act: index into neighbors of current (masked)
    Reward: -(blended edge cost)/scale + small phi shaping - small penalties + goal bonus
    Masks:
      - Only allow moves that can still reach destination (reverse reachability)
      - Avoid revisiting nodes unless no alternative (soft)
    """
    metadata = {"render_modes": ["human"]}

    def __init__(self, G: nx.DiGraph, origin: str, destination: str,
                 phys_edges: Dict[Tuple[str,str], Dict[str,float]],
                 objective: str, blend_alpha: float, scaling: float, seed: int = 0):
        super().__init__()
        self.G, self.origin, self.destination = G, origin, destination
        self.phys_edges = phys_edges
        self.objective = objective
        self.blend_alpha = float(np.clip(blend_alpha, 0.0, 1.0))
        self.scale = float(max(scaling, 1e-6))
        self.rng = np.random.default_rng(seed)

        # nodes & neighbors
        self.nodes = list(G.nodes()); assert origin in self.nodes and destination in self.nodes
        self.n = len(self.nodes)
        self.node2id = {n:i for i,n in enumerate(self.nodes)}
        self.id2node = {i:n for n,i in self.node2id.items()}
        self.neigh = {i: [self.node2id[v] for v in G.successors(u)] for i,u in enumerate(self.nodes)}
        self.max_deg = max(1, max((len(v) for v in self.neigh.values()), default=1))

        # step cap
        try:
            hops = nx.shortest_path_length(G.to_undirected(), origin, destination)
        except Exception:
            hops = 20
        self.max_steps = int(min(5*max(2,hops)+10, 300))

        # destination potential & reachability set
        self.phi = precompute_phi(G, destination)
        self.reach_ok = nodes_that_can_reach_dest(G, destination)  # key part: mask detours away from goal
        self._dest_id = self.node2id[self.destination]

        # gym spaces
        self.observation_space = spaces.Box(0.0, 1.0, shape=(2*self.n+1,), dtype=np.float32)
        self.action_space = spaces.Discrete(self.max_deg)

        # state
        self.current = None; self.prev = None; self.steps = 0
        self.visit_counts = None
        self.kpi_totals = None
        self.pred_sum = 0.0

    def _obs(self):
        x = np.zeros(2*self.n+1, dtype=np.float32)
        x[self.node2id[self.current]] = 1.0
        x[self.n + self._dest_id] = 1.0
        x[-1] = float(self.phi.get(self.current, 1.0))
        return x

    def _objective_kpi_cost(self, u, v):
        t, m, c = edge_cost_components(self.phys_edges, u, v)
        if self.objective == "fastest": return t
        if self.objective == "cheapest": return m
        return c

    def _blended_cost(self, u, v):
        kpi_cost = self._objective_kpi_cost(u, v)
        pred_w   = float(self.G[u][v].get("weight", 0.0))
        pred_w_n = pred_w / (1.0 + abs(pred_w))  # squashed for stability
        return (1.0 - self.blend_alpha)*kpi_cost + self.blend_alpha*pred_w_n

    def get_action_mask(self) -> np.ndarray:
        nid = self.node2id[self.current]
        neigh = self.neigh[nid]
        mask = np.zeros(self.max_deg, dtype=bool)
        if not neigh:
            mask[0] = True
            return mask

        # 1) keep only neighbors that can reach destination
        candidates = [i for i,nbr in enumerate(neigh) if self.id2node[nbr] in self.reach_ok]

        # 2) avoid revisits among those (soft)
        non_revisit = [i for i in candidates if self.visit_counts[self.id2node[neigh[i]]] == 0]

        used = None
        if non_revisit:
            used = non_revisit
        elif candidates:
            used = candidates
        else:
            # if none can reach destination (disconnected region), allow all neighbors but prefer non-revisit
            nonrev_all = [i for i in range(len(neigh)) if self.visit_counts[self.id2node[neigh[i]]] == 0]
            used = nonrev_all if nonrev_all else list(range(len(neigh)))

        mask[:len(neigh)] = False
        for i in used:
            mask[i] = True
        return mask

    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        if seed is not None: self.rng = np.random.default_rng(seed)
        self.current = self.origin; self.prev = None; self.steps = 0
        self.visit_counts = defaultdict(int); self.visit_counts[self.origin] = 1
        self.kpi_totals = defaultdict(float); self.pred_sum = 0.0
        return self._obs(), {}

    def step(self, action: int):
        self.steps += 1
        info = {}

        neigh = self.neigh[self.node2id[self.current]]
        if not neigh:
            return self._obs(), -5.0, False, True, info
        if action >= len(neigh):
            return self._obs(), -1.0, False, (self.steps >= self.max_steps), info

        next_id = neigh[action]
        u, v = self.current, self.id2node[next_id]

        # reward (objective-aligned & blended)
        cost = self._blended_cost(u, v)
        reward = -(cost / self.scale)

        # small shaping & penalties
        phi_s, phi_sp = float(self.phi.get(u,1.0)), float(self.phi.get(v,1.0))
        reward += SHAPING_ETA * (phi_s - phi_sp)
        reward -= STEP_PENALTY
        if self.prev is not None and v == self.prev:
            reward -= BACKTRACK_P

        # anti-loop
        self.visit_counts[v] += 1
        if self.visit_counts[v] > 1:
            reward -= REVISIT_P
        if self.visit_counts[v] > REVISIT_LIMIT:
            reward -= 5.0
            info.update({f"sum_{k}": val for k, val in self.kpi_totals.items()})
            info["sum_pred_weight"] = self.pred_sum
            return self._obs(), reward, False, True, info

        # KPI accumulation (for reporting)
        pe = self.phys_edges.get((u, v), {})
        for k in ["distance_km","travel_time_hr","fuel_cost_gbp","co2_kg","toll_gbp",
                  "caz_charge_gbp","hgv_levy_gbp","lez_cost_gbp","traffic_delay_hr","weather_delay_hr"]:
            self.kpi_totals[k] += float(pe.get(k, 0.0))
        self.pred_sum += float(self.G[u][v].get("weight", 0.0))

        # state update
        self.prev, self.current = self.current, v
        terminated = (self.current == self.destination)
        truncated  = (self.steps >= self.max_steps)
        if terminated: reward += GOAL_BONUS

        info.update({f"sum_{k}": val for k, val in self.kpi_totals.items()})
        info["sum_pred_weight"] = self.pred_sum
        return self._obs(), reward, terminated, truncated, info

# ----------------------- Deterministic beam finisher --------------------------
def beam_finish(base_env: RoutingEnv, k: int = BEAM_K, max_steps: int = BEAM_STEPS):
    """
    Bounded deterministic search over blended cost (NOT A*/Dijkstra).
    Respects reachability set; tie-breaks with phi.
    Returns: list of (u,v) edges to append, and boolean reached_dest.
    """
    start = base_env.current
    Node = tuple  # (cum_cost, phi, node, path_nodes)
    # priority by (cum_cost, phi)
    beams: List[Node] = [(0.0, base_env.phi.get(start,1.0), start, [start])]
    seen_best = {start: 0.0}
    steps = 0

    def expand(node: Node):
        cum, _, u, path = node
        out = []
        for nid in base_env.neigh[base_env.node2id[u]]:
            v = base_env.id2node[nid]
            # prefer only nodes that can reach destination
            if v not in base_env.reach_ok:
                continue
            if v in path:
                continue  # avoid cycles in finisher
            c = base_env._blended_cost(u, v)
            new_cum = cum + c
            if new_cum >= seen_best.get(v, float("inf")) - 1e-9:
                continue
            seen_best[v] = new_cum
            out.append((new_cum, base_env.phi.get(v,1.0), v, path + [v]))
        # if nothing legal and u has neighbors, allow non-reach_ok as last resort
        if not out:
            for nid in base_env.neigh[base_env.node2id[u]]:
                v = base_env.id2node[nid]
                if v in path: continue
                c = base_env._blended_cost(u, v)
                new_cum = cum + c
                if new_cum >= seen_best.get(v, float("inf")) - 1e-9:
                    continue
                seen_best[v] = new_cum
                out.append((new_cum, base_env.phi.get(v,1.0), v, path + [v]))
        return out

    while beams and steps < max_steps:
        steps += 1
        # expand each beam, collect candidates
        cand = []
        for b in beams:
            if b[2] == base_env.destination:  # already at dest
                path = b[3]
                # convert to edges
                tail = [(path[i], path[i+1]) for i in range(len(path)-1)]
                return tail, True
            cand.extend(expand(b))
        if not cand:
            break
        # keep k best by (cum_cost, phi)
        cand.sort(key=lambda z: (z[0], z[1]))
        beams = cand[:k]

    # best partial if not reached
    best = min(beams, key=lambda z: (z[0], z[1])) if beams else None
    if best is None:
        return [], False
    path = best[3]
    tail = [(path[i], path[i+1]) for i in range(len(path)-1)]
    return tail, (path[-1] == base_env.destination)

# -------------------------- rollout & reporting -------------------------------
def rollout(model, env, render: bool = False):
    base = env
    while hasattr(base, "env"): base = base.env
    obs, _ = env.reset()
    path = [base.current]; total = 0.0
    term = trunc = False; last = {}
    while not (term or trunc):
        mask = base.get_action_mask()
        a, _ = model.predict(obs, action_masks=mask)
        obs, r, term, trunc, info = env.step(int(a))
        total += float(r); path.append(base.current); last = info
        if render: print(base.current)

    # finish if needed
    if base.current != base.destination:
        tail, ok = beam_finish(base, k=BEAM_K, max_steps=BEAM_STEPS)
        for u, v in tail:
            path.append(v)
            pe = base.phys_edges.get((u, v), {})
            for k in ["distance_km","travel_time_hr","fuel_cost_gbp","co2_kg","toll_gbp",
                      "caz_charge_gbp","hgv_levy_gbp","lez_cost_gbp","traffic_delay_hr","weather_delay_hr"]:
                last[f"sum_{k}"] = last.get(f"sum_{k}", 0.0) + float(pe.get(k, 0.0))
            last["sum_pred_weight"] = last.get("sum_pred_weight", 0.0) + float(base.G[u][v].get("weight", 0.0))

    kpis = {k.replace("sum_", ""): v for k, v in last.items() if str(k).startswith("sum_")}
    pred = float(last.get("sum_pred_weight", 0.0))
    return path, total, kpis, pred

def report(title, path, kpis, pred=None):
    print(f"{title}: {path}")
    if pred is not None:
        print(f"Predicted blended weight (approx): {pred:.3f}")
    print("KPIs:", {k: round(v, 3) for k, v in kpis.items()})

# ------------------------------- training ------------------------------------
def train_for_objective(G, origin, destination, objective, seed=0, steps1=60_000, steps2=30_000):
    scale = normalize_scale(G, phys_edges, objective)

    def mk_env():
        base = RoutingEnv(G, origin, destination, phys_edges,
                          objective=objective, blend_alpha=BLEND_ALPHA,
                          scaling=scale, seed=seed)
        base.phi = precompute_phi(G, destination)  # cache φ
        return ActionMasker(base, lambda e: e.get_action_mask())

    vec = make_vec_env(mk_env, n_envs=8, seed=seed)
    model = MaskablePPO("MlpPolicy", vec, verbose=0,
                        learning_rate=3e-4, n_steps=1024, batch_size=128,
                        gamma=0.995, ent_coef=0.001, seed=seed)
    model.learn(total_timesteps=steps1)
    # light finetune
    model.set_env(vec); model.learn(total_timesteps=steps2)

    test_env = mk_env()
    path, ret, kpis, pred = rollout(model, test_env, render=False)
    return {"model": model, "path": path, "kpis": kpis, "pred": pred}

# ===== Train once for your chosen O/D (change these as needed) ===============
origin = "Birmingham Central"
destination = "Teesport"

trained = {}
for obj in OBJECTIVES:
    print("\n" + "="*60)
    print(f"Training objective: {obj}")
    out = train_for_objective(G_pred[obj], origin, destination, obj, seed=0)
    trained[obj] = out["model"]
    report(f"[FINAL] {obj}", out["path"], out["kpis"], out["pred"])

# ===== Fast inference (no retraining) ========================================
def route_request(objective: str, origin: str, destination: str):
    model = trained[objective]
    G = G_pred[objective]
    scale = normalize_scale(G, phys_edges, objective)
    base = RoutingEnv(G, origin, destination, phys_edges,
                      objective=objective, blend_alpha=BLEND_ALPHA,
                      scaling=scale, seed=123)
    base.phi = precompute_phi(G, destination)
    env = ActionMasker(base, lambda e: e.get_action_mask())
    path, _, kpis, pred = rollout(model, env, render=False)
    return {"path": path, "kpis": kpis, "pred": pred}

# Example:
# res = route_request("fastest", "Birmingham Central", "Carlisle")
# print(res)



Training objective: cheapest
[FINAL] cheapest: ['Birmingham Central', 'Watford Gap', 'East Midlands Gateway', 'Stoke-on-Trent', 'Teesport']
Predicted blended weight (approx): 2615.510
KPIs: {'distance_km': 463.296, 'travel_time_hr': 7.273, 'fuel_cost_gbp': 221.163, 'co2_kg': 400.486, 'toll_gbp': 5.853, 'caz_charge_gbp': 30.018, 'hgv_levy_gbp': 0.0, 'lez_cost_gbp': 0.0, 'traffic_delay_hr': 0.213, 'weather_delay_hr': 0.071, 'pred_weight': 2615.51}

Training objective: fastest
[FINAL] fastest: ['Birmingham Central', 'ProLogis Park DC', 'Stoke-on-Trent', 'Teesport']
Predicted blended weight (approx): 17.827
KPIs: {'distance_km': 338.024, 'travel_time_hr': 5.162, 'fuel_cost_gbp': 156.96, 'co2_kg': 284.229, 'toll_gbp': 6.718, 'caz_charge_gbp': 45.0, 'hgv_levy_gbp': 0.0, 'lez_cost_gbp': 0.0, 'traffic_delay_hr': 0.206, 'weather_delay_hr': 0.101, 'pred_weight': 17.827}

Training objective: greenest
[FINAL] greenest: ['Birmingham Central', 'ProLogis Park DC', 'Stoke-on-Trent', 'Teesport']
Predi

In [1]:
import pandas as pd

def evaluate_models(G_pred, trained, phys_edges, od_pairs, objectives=OBJECTIVES):
    results = []
    for obj in objectives:
        model = trained[obj]
        G = G_pred[obj]
        for (origin, destination) in od_pairs:
            scale = normalize_scale(G, phys_edges, obj)
            base = RoutingEnv(G, origin, destination, phys_edges,
                              objective=obj, blend_alpha=BLEND_ALPHA,
                              scaling=scale, seed=42)
            env = ActionMasker(base, lambda e: e.get_action_mask())
            path, _, kpis, _ = rollout(model, env, render=False)
            results.append({
                "objective": obj,
                "origin": origin,
                "destination": destination,
                "distance_km": kpis.get("distance_km", 0),
                "travel_time_hr": kpis.get("travel_time_hr", 0),
                "total_cost_gbp": kpis.get("fuel_cost_gbp", 0)
                                   + kpis.get("toll_gbp", 0)
                                   + kpis.get("caz_charge_gbp", 0)
                                   + kpis.get("hgv_levy_gbp", 0),
                "co2_kg": kpis.get("co2_kg", 0),
                "reward": np.mean(list(kpis.values())),
            })
    df = pd.DataFrame(results)
    df["efficiency_index"] = 1 / (df["travel_time_hr"] * df["total_cost_gbp"] * df["co2_kg"] + 1e-6)
    return df

# Example OD pairs
od_pairs = [
    ("Birmingham Central", "Teesport"),
    ("Manchester Depot", "Hull Port"),
    ("Leeds Terminal", "Immingham Dock")
]

eval_df = evaluate_models(G_pred, trained, phys_edges, od_pairs)
display(eval_df)


NameError: name 'OBJECTIVES' is not defined